# SpaBound — Mouse embryo: RNA + ATAC

This example keeps the source notebook's active preprocessing, graph and training settings. This is a runnable workflow example, not a claim of exact paper-table reproduction.

Provide your own paired H5AD files; data and previously generated results are not included. Install the package and notebook dependencies as described in the repository README, including R/mclust for clustering. Run all cells in order from a fresh kernel. Data layout and configuration notes are in [examples/README.md](README.md).

## 1. Imports, seeds and paths

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, homogeneity_score,
)

from spabound.preprocessing import pca, lsi, clr_normalize_each_cell
from spabound.graph_v2 import construct_boundary_aware_graphs_v2
from spabound.trainer import SpaBound
from spabound.utils import clustering


def setup_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Run with the working directory at the repository root or examples/.
cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / 'pyproject.toml').is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Start Jupyter in the SpaBound repository root or examples/.')


def configured_root(variable, default_folder):
    value = Path(os.environ.get(variable, default_folder)).expanduser()
    return value if value.is_absolute() else REPO_ROOT / value


DATA_ROOT = configured_root('SPABOUND_DATA_DIR', 'data')
OUTPUT_ROOT = configured_root('SPABOUND_OUTPUT_DIR', 'outputs')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PREPROCESS_SEED = 42
setup_seed(PREPROCESS_SEED)

## 2. Configuration and paired inputs

`SPABOUND_DATA_DIR` selects a root containing the dataset subfolders; it defaults to `data/` in the repository. `SPABOUND_OUTPUT_DIR` defaults to `outputs/`. Relative environment-variable paths are resolved from the repository root.

The inputs must have raw counts in `X`, matching unique spot identifiers in the same order, and spatial coordinates in `obsm['spatial']`. Reference labels are optional and are used only for the metric/plot cells here. Cluster counts are explicit example settings. The trainer seed is 2022; preprocessing uses 42 and the mclust helper uses R seed 2020.

In [ ]:
DATASET = 'mouse_embryo'
DATA_DIR = DATA_ROOT / DATASET
OUTPUT_DIR = OUTPUT_ROOT / DATASET
RNA_PATH = DATA_DIR / 's1_adata_rna.h5ad'
MODALITY2_PATH = DATA_DIR / 's1_adata_atac.h5ad'
REFERENCE_KEY = 'Joint_clusters'
N_CLUSTERS = 14

GRAPH_CONFIG = {
    'n_neighbors_spatial': 6,
    'sigma_spatial': None,
    'beta_spatial': 0.8,
    'spatial_candidates_k': 40,
    'n_neighbors_feature': 15,
    'beta_feature': 0.6,
    'verbose': True,
}

TRAINING_CONFIG = {
    'datatype': 'SPOTS',
    'learning_rate': 0.0002,
    'weight_decay': 0.0001,
    'dim_output': 160,
    'z_dim': 64,
    'n_latent_shared': 32,
    'n_latent_private': 16,
    'mmvae_weight': 1.0,
    'reconstruction_weight': 1.0,
    'kl_weight': 0.005,
    'edge_preserving_weight': 0.0001,
    'use_appnp': True,
    'appnp_hidden': 128,
    'appnp_layers': 2,
    'appnp_alpha': 0.25,
    'appnp_residual': True,
    'appnp_dropout': 0.0,
    'use_boundary_aware_graph': True,
    'use_edge_preserving_loss': True,
    'edge_preserving_type': 'charbonnier',
    'epochs': 300,
    'random_seed': 2022,
}

In [ ]:
for input_path in (RNA_PATH, MODALITY2_PATH):
    if not input_path.is_file():
        raise FileNotFoundError(
            f'Missing {input_path}. Supply the paired H5AD data; see examples/README.md.'
        )
rna_raw = sc.read_h5ad(RNA_PATH)
modality2_raw = sc.read_h5ad(MODALITY2_PATH)
rna_raw.var_names_make_unique()
modality2_raw.var_names_make_unique()
if not rna_raw.obs_names.is_unique or not modality2_raw.obs_names.is_unique:
    raise ValueError('Spot identifiers must be unique in both modalities.')
if not rna_raw.obs_names.equals(modality2_raw.obs_names):
    raise ValueError('The two H5AD files must contain the same spots in the same order.')
for adata, name in ((rna_raw, 'RNA'), (modality2_raw, 'second modality')):
    if 'spatial' not in adata.obsm:
        raise ValueError(f'{name} requires coordinates in obsm["spatial"].')
    if not np.isfinite(np.asarray(adata.obsm['spatial'])).all():
        raise ValueError(f'{name} spatial coordinates must be finite.')
print(f'Loaded {rna_raw.n_obs} paired spots; device: {DEVICE}')

## 3. Modality-specific preprocessing

In [ ]:
# Reset from fresh-loaded objects so this cell can be rerun safely.
setup_seed(PREPROCESS_SEED)
adata_omics1 = rna_raw.copy()
adata_omics2 = modality2_raw.copy()

sc.pp.filter_genes(adata_omics1, min_cells=10)
sc.pp.highly_variable_genes(adata_omics1, flavor='seurat_v3', n_top_genes=3000)
sc.pp.normalize_total(adata_omics1, target_sum=1e4)
sc.pp.log1p(adata_omics1)
sc.pp.scale(adata_omics1)
adata_high = adata_omics1[:, adata_omics1.var['highly_variable']]

# Retain the source notebook's RNA PCA dimension rule.
adata_omics1.obsm['feat'] = pca(adata_high, n_comps=min(50, adata_omics2.n_vars - 1))
# Reuse supplied LSI if present; otherwise calculate 51 components and drop the first.
if 'X_lsi' not in adata_omics2.obsm:
    sc.pp.highly_variable_genes(adata_omics2, flavor='seurat_v3', n_top_genes=3000)
    lsi(adata_omics2, use_highly_variable=False, n_components=51)
adata_omics2.obsm['feat'] = adata_omics2.obsm['X_lsi'].copy()

## 4. Boundary-aware graph construction

In [ ]:
setup_seed(PREPROCESS_SEED)
graphs = construct_boundary_aware_graphs_v2(
    adata_omics1, adata_omics2, **GRAPH_CONFIG
)
data = {'adata_omics1': adata_omics1, 'adata_omics2': adata_omics2}
for key in (
    'adj_spatial_omics1', 'adj_spatial_omics2',
    'adj_feature_omics1', 'adj_feature_omics2',
):
    data[key] = graphs[key]
data['edge_gradients'] = graphs.get('edge_gradients')

## 5. Train SpaBound

In [ ]:
setup_seed(PREPROCESS_SEED)
trainer = SpaBound(data=data, device=DEVICE, **TRAINING_CONFIG)
output = trainer.train()

## 6. Cluster the shared representation

`output['SpaBound']` contains the row-normalized shared posterior-mean representation. Clustering applies PCA20 followed by mclust with the stated cluster count. Auxiliary private coordinates are not interpreted as validated modality-specific biological factors.

In [ ]:
setup_seed(PREPROCESS_SEED)
result = adata_omics1.copy()
result.obsm['SpaBound'] = output['SpaBound']
clustering(
    result, key='SpaBound', add_key='SpaBound',
    n_clusters=N_CLUSTERS, method='mclust', use_pca=True, n_comps=20,
)
result.obs['SpaBound'] = result.obs['SpaBound'].astype('category')

## 7. Evaluate and visualize

ARI, NMI, AMI and homogeneity do not require aligning cluster identifiers. Missing reference labels are excluded from metrics and the evaluated count is reported. Predictions are not relabeled against the reference.

In [ ]:
metrics = {'n_spots': int(result.n_obs)}
if REFERENCE_KEY in result.obs:
    valid = result.obs[REFERENCE_KEY].notna() & result.obs['SpaBound'].notna()
    metrics['n_evaluated'] = int(valid.sum())
    if not valid.any():
        raise ValueError('No spots have both reference and predicted labels.')
    truth = result.obs.loc[valid, REFERENCE_KEY].astype(str)
    prediction = result.obs.loc[valid, 'SpaBound'].astype(str)
    metrics.update(
        ARI=adjusted_rand_score(truth, prediction),
        NMI=normalized_mutual_info_score(truth, prediction),
        AMI=adjusted_mutual_info_score(truth, prediction),
        homogeneity=homogeneity_score(truth, prediction),
    )
else:
    print(f'No {REFERENCE_KEY!r} column: skipping reference-label metrics.')
pd.Series(metrics, name=DATASET)

In [ ]:
sc.pp.neighbors(result, use_rep='SpaBound')
sc.tl.umap(result)
colors = ([REFERENCE_KEY] if REFERENCE_KEY in result.obs else []) + ['SpaBound']
sc.pl.umap(result, color=colors, size=25)
sc.pl.embedding(result, basis='spatial', color=colors, size=50)

## 8. Save outputs

Write a separate result H5AD, cluster labels, metrics and the settings used for this run. Input H5AD files are never overwritten. Rerunning this cell replaces the example's previous files in its output subfolder.

In [ ]:
import json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Preserve the input RNA matrix; add results to a separate output file.
export = rna_raw.copy()
export.obs['SpaBound'] = result.obs['SpaBound'].astype(str).to_numpy()
export.obsm['SpaBound'] = result.obsm['SpaBound'].copy()
export.obsm['X_umap_SpaBound'] = result.obsm['X_umap'].copy()
export.write_h5ad(OUTPUT_DIR / 'spabound_result.h5ad')
result.obs[['SpaBound']].to_csv(OUTPUT_DIR / 'clusters.csv', index_label='spot_id')
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
configuration = {
    'dataset': DATASET,
    'preprocess_seed': PREPROCESS_SEED,
    'mclust_seed': 2020,
    'n_clusters': N_CLUSTERS,
    'graph': GRAPH_CONFIG,
    'training': TRAINING_CONFIG,
}
(OUTPUT_DIR / 'configuration.json').write_text(
    json.dumps(configuration, indent=2) + '\n', encoding='utf-8'
)
print(f'Results saved in {OUTPUT_DIR}')